# Data Loss Prevention  

### Note:
The built-in infoType detectors are not a 100% accurate detection method. For example, they can't guarantee compliance with regulatory requirements. You must decide what data is sensitive and how to best protect it. Google recommends that you test your settings to make sure your configuration meets your requirements.

In [1]:
import pandas as pd

from vertexai.preview.language_models import TextGenerationModel
import google.cloud.dlp
import google.cloud.dlp_v2

from dsgcp import credentials, bq, sheets, dlp

pd.options.display.max_columns = None
pd.options.display.max_rows = None
pd.options.display.max_colwidth = 1000

In [2]:
%store -r cred_file
%store -r project_id
%store -r dataset_id

In [3]:
(gcp_credentials, 
 project_id) = credentials.create_credentials(cred_file=cred_file, 
                                              project_id=project_id)

In [4]:
query_text = """
SELECT *
FROM `corp-skyrise-dev.octo.20230713_snow_cases`
LIMIT 100
"""
df_snow = bq.basic_query(gcp_credentials, query_text)

BigQuery query bytes:   0%|          | 0/84677882 [00:00<?, ?it/s]


In [5]:
df_snow.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 71 columns):
 #   Column                        Non-Null Count  Dtype              
---  ------                        --------------  -----              
 0   account                       0 non-null      object             
 1   account_id                    100 non-null    object             
 2   approval                      100 non-null    object             
 3   assigned_to                   91 non-null     object             
 4   assigned_to_id                91 non-null     object             
 5   assigned_on_timestamp         91 non-null     datetime64[ns, UTC]
 6   assignment_group              100 non-null    object             
 7   assignment_group_id           100 non-null    object             
 8   case_name                     100 non-null    object             
 9   case_report                   100 non-null    object             
 10  case_type                     96 non-nu

In [5]:
dlp_service = dlp.DLPContent()
dlp_service.project_id = 'sada-colin-dietrich'

In [6]:
dfx = pd.DataFrame([
    ['James', 'Kirk', '3/22/2233', 'Type A', 'A town in Ohio'],
    ['James', 'Potter', '3/27/1960', 'Type B', 'A city in England'],
    ['William', 'Riker', '8/19/2335', 'Type A-', 'A cabin in Alaska']
], columns=['first_name', 'last_name', 'dob', 'blood_type', 'birth_context'])

In [7]:
dfx

,first_name,last_name,dob,blood_type,birth_context
0,James,Kirk,3/22/2233,Type A,A town in Ohio
1,James,Potter,3/27/1960,Type B,A city in England
2,William,Riker,8/19/2335,Type A-,A cabin in Alaska


In [8]:
dlp_service.deidentify_df(dfx)

,first_name,last_name,dob,blood_type,birth_context
0,[PERSON_NAME],[PERSON_NAME],3/22/2233,Type A,A town in [LOCATION]
1,[PERSON_NAME],[PERSON_NAME],3/27/1960,Type B,A city in [LOCATION]
2,[PERSON_NAME],[PERSON_NAME],8/19/2335,Type A-,A cabin in [LOCATION]


### DLP Info Types  
Types that can be scanned for and replaced

In [9]:
dfit = dlp_service.info_types_df()
dfit.head()

,api_string,display_name,description,industry_category,location_category,type_category
0,DOCUMENT_TYPE/LEGAL/PLEADING,Legal pleading,"A pleading is a formal written statement of a party's claims or defenses to another party's claims. Examples include a complaint, a demurrer, or an answer.",None,GLOBAL,DOCUMENT
1,DOCUMENT_TYPE/LEGAL/BRIEF,Legal brief,"A legal brief is a document advocating a particular outcome of the case, presenting supporting points, law interpretations, and recommendations.",None,GLOBAL,DOCUMENT
2,CREDIT_CARD_TRACK_NUMBER,Credit card track number,A credit card track number is a variable length alphanumeric string. It is used to store key cardholder information.,FINANCE,GLOBAL,SPII
3,MEXICO_CURP_NUMBER,Mexico population registry number,"The Mexico Clave Única de Registro de Población (CURP) number, or Unique Population Registry Code or Personal Identification Code number. This is an 18-character state-issued identification number assigned by the Mexican government to citizens or residents of Mexico and used for taxpayer identification.",None,MEXICO,GOVERNMENT_ID
4,ORGANIZATION_NAME,Organization name,"A name of a chain store, business or organization. Note: Not recommended for use during latency sensitive operations.",None,GLOBAL,CONTEXTUAL_INFORMATION


In [10]:
dfit.location_category.unique()

array(['GLOBAL', 'MEXICO', 'KOREA', 'GERMANY', 'HONG_KONG', 'FRANCE',
       'CANADA', 'IRELAND', 'UNITED_STATES', 'AUSTRALIA', 'PERU',
       'COLOMBIA', 'SPAIN', 'DENMARK', 'INDONESIA', 'BRAZIL',
       'NEW_ZEALAND', 'FINLAND', 'SINGAPORE', 'PORTUGAL', 'CHILE',
       'THAILAND', 'UNITED_KINGDOM', 'CHINA', 'JAPAN', 'ARGENTINA',
       'THE_NETHERLANDS', 'INDIA', 'BELGIUM', 'ITALY', 'POLAND',
       'SOUTH_AFRICA', 'SWEDEN', 'VENEZUELA', 'PARAGUAY', 'TAIWAN',
       'NORWAY', 'INTERNAL', 'URUGUAY', 'TURKEY', '42', 'ISRAEL'],
      dtype=object)

In [11]:
dfit.industry_category.unique()

array([None, 'FINANCE', 'TELECOMMUNICATIONS', 'HEALTH'], dtype=object)

### Deidentify with Replace

In [12]:
test_value = "My name is Jim and my Social Security Number is 531-554-9874"

dlp_service.deidentify_with_replace(value=test_value,
                             replacement='foo',
                             info_types=['MEXICO_CURP_NUMBER'],
                             industry_category=['FINANCE'],
                             location_category=['MEXICO'],
                             type_category=['GOVERNMENT_INFORMATION'])

'My name is Jim and my Social Security Number is 531-554-9874'

In [13]:
test_value = "My name is Jim and my Social Security Number is 531-554-9874"

dlp_service.deidentify_with_replace(value=test_value,
                             replacement='TOS',
                             location_category=['GLOBAL'],
                             info_types=None)

'My name is TOS and my Social Security Number is TOS'

In [14]:
test_value = "My name is Jim and my Starfleet Registry number is 531-554-9874"

dlp_service.deidentify_with_replace(value=test_value,
                             replacement='TOS',
                             location_category=['GLOBAL'],
                             info_types=None)

'My name is TOS and my Starfleet Registry number is TOS'

In [15]:
test_value = "My name is Spock and my Social Security Number is 536-332-9877"

dlp_service.deidentify_with_replace(value=test_value, replacement='TOS')

'My name is TOS and my Social Security Number is TOS'

In [16]:
test_value = "My name is Spock and my Social Security Number is 536-332-9877"

dlp_service.deidentify_with_replace(value=test_value)

'My name is [PERSON_NAME] and my Social Security Number is [PHONE_NUMBER]'

In [17]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

### BQ DLP - Does not yet work

In [ ]:
dlpc = google.cloud.dlp_v2.DlpServiceClient()

In [ ]:
info_types = None
custom_dictionaries = None
custom_regexes = None
min_likelihood = 'POSSIBLE'
max_findings = 10
timeout = 10

project_id = 'sada-colin-dietrich'

# first test
dataset_id = 'skus'
table_id   = 'current'

# second test
dataset_id = 'test_01'
table_id   = 'pii_02'

In [ ]:
# Prepare info_types by converting the list of strings into a list of
# dictionaries (protos are also accepted).
if not info_types:
    info_types = ["FIRST_NAME", "LAST_NAME", "EMAIL_ADDRESS"]
info_types = [{"name": info_type} for info_type in info_types]

In [ ]:
info_types

In [ ]:
# Prepare custom_info_types by parsing the dictionary word lists and
# regex patterns.
if custom_dictionaries is None:
    custom_dictionaries = []
dictionaries = [
    {
        "info_type": {"name": f"CUSTOM_DICTIONARY_{i}"},
        "dictionary": {"word_list": {"words": custom_dictionaries.split(",")}},
    }
    for i, custom_dict in enumerate(custom_dictionaries)
]

In [ ]:
if custom_regexes is None:
    custom_regexes = []
regexes = [
    {
        "info_type": {"name": f"CUSTOM_REGEX_{i}"},
        "regex": {"pattern": custom_regex},
    }
    for i, custom_regex in enumerate(custom_regexes)
]

In [ ]:
custom_info_types = dictionaries + regexes

In [ ]:
# Construct the configuration dictionary. Keys which are None may
# optionally be omitted entirely.
inspect_config = {
    "info_types": info_types,
    "custom_info_types": custom_info_types,
    "min_likelihood": min_likelihood,
    "limits": {"max_findings_per_request": max_findings},
}

In [ ]:
# Construct a storage_config containing the target Bigquery info.
storage_config = {
    "big_query_options": {
        "table_reference": {
            "project_id": project_id,  # bigquery_project_id
            "dataset_id": dataset_id,
            "table_id": table_id,
        }
    }
}

In [ ]:
parent = f"projects/{project_id}/locations/global"

In [ ]:
# Construct the inspect_job, which defines the entire inspect content task.
inspect_job = {
    "inspect_config": inspect_config,
    "storage_config": storage_config,
    #"actions": actions,
}

In [ ]:
operation = dlpc.create_dlp_job(
    request={"parent": parent, "inspect_job": inspect_job}
    )
print(f"Inspection operation started: {operation.name}")

In [ ]:
operation.state

In [ ]:
job_states = [
['JOB_STATE_UNSPECIFIED', 0, 'Unused.'],
['PENDING',1,'The job has not yet started.'],
['RUNNING',2,'The job is currently running. Once a job has finished it will transition to FAILED or DONE.'],
['DONE',3,'The job is no longer running.'],
['CANCELED',4,'The job was canceled before it could be completed.'],
['FAILED',5,'The job had an error and did not complete.'],
['ACTIVE',6,'The job is currently accepting findings via hybridInspect. A hybrid job in ACTIVE state may continue to have findings added to it through the calling of hybridInspect. After the job has finished no more calls to hybridInspect may be made. ACTIVE jobs can transition to DONE.']
]

In [ ]:
pd.DataFrame(job_states, columns=['id', 'number', 'description'])

In [ ]:
job = dlpc.get_dlp_job(request={"name": operation.name})

In [ ]:
job.state

In [ ]:
type(job.state)

In [ ]:
job.inspect_details.result.info_type_stats

In [ ]:
if job.inspect_details.result.info_type_stats:
    for finding in job.inspect_details.result.info_type_stats:
        print(
            "Info type: {}; Count: {}".format(
                finding.info_type.name, finding.count
            )
        )

In [ ]:
job.inspect_details